RQ2 TP and FN Analysis


In [1]:
import pandas as pd
import os
from pathlib import Path
# -----------------------------
# File paths
# -----------------------------
COMMENTS_FILE = "../data/comment.csv"

for pred_file in Path("../result/rq2/unique").rglob("*.csv"):


    output_satd_all_file = f"../cache/output/analysis/satd_all_{pred_file.name}"
    output_tp_file = f"../cache/output/analysis/satd_tp_{pred_file.name}"
    output_fn_file = f"../cache/output/analysis/satd_fn_{pred_file.name}"
    os.makedirs(os.path.dirname(output_satd_all_file), exist_ok=True)

    # -----------------------------
    # Load data
    # -----------------------------
    comments = pd.read_csv(COMMENTS_FILE)
    pred = pd.read_csv(pred_file)

    # -----------------------------
    # Merge predictions with ground truth
    # -----------------------------
    df = comments.merge(
        pred[['id', 'label_pred']],
        on="id",
        how="inner"
    )

    # -----------------------------
    # Convert to binary SATD detection task
    # -----------------------------
    df["is_satd"] = df["satd"] == "yes"
    df["pred_satd"] = df["label_pred"] == "yes"

    # -----------------------------
    # Compute confusion matrix label
    # -----------------------------
    df["confusion"] = "TN"

    df.loc[(df.is_satd) & (df.pred_satd), "confusion"] = "TP"
    df.loc[(df.is_satd) & (~df.pred_satd), "confusion"] = "FN"
    df.loc[(~df.is_satd) & (df.pred_satd), "confusion"] = "FP"
    df.loc[(~df.is_satd) & (~df.pred_satd), "confusion"] = "TN"

    # -----------------------------
    # SATD-only dataset
    # -----------------------------
    satd_df = df[df["is_satd"]].copy()

    # -----------------------------
    # Compute statistics per SATD type
    # -----------------------------
    stats = (
        satd_df
        .groupby(["type", "confusion"])
        .size()
        .unstack(fill_value=0)
    )

    # Ensure required columns exist
    for c in ["TP", "FN"]:
        if c not in stats.columns:
            stats[c] = 0

    # Compute totals and percentages
    stats["total_satd"] = stats["TP"] + stats["FN"]
    stats["correct"] = stats["TP"]
    stats["failed"] = stats["FN"]

    stats["correct_%"] = (stats["correct"] / stats["total_satd"] * 100).round(2)
    stats["failed_%"] = (stats["failed"] / stats["total_satd"] * 100).round(2)

    stats = stats.sort_values("total_satd", ascending=False)

    print("\n", f"{'#' *10 }{pred_file.stem}{'#' *10 }", end="\n")
    print(stats)


    # -----------------------------
    # Export SATD-only CSV with confusion labels
    # -----------------------------
    output_cols = list(comments.columns) + ["label_pred", "confusion"]

    satd_output = satd_df[output_cols]

    satd_output.to_csv(
        output_satd_all_file,
        index=False
    )


    # -----------------------------
    # Export correctly detected SATDs
    # -----------------------------
    satd_output[satd_output["confusion"] == "TP"].to_csv(
        output_tp_file,
        index=False
    )



    # -----------------------------
    # Export missed SATDs
    # -----------------------------
    satd_output[satd_output["confusion"] == "FN"].to_csv(
        output_fn_file,
        index=False
    )




 ##########detect_trained-liu-detector-5fcv-1##########
confusion             FN  TP  total_satd  correct  failed  correct_%  failed_%
type                                                                          
composite              4   4           8        4       4      50.00     50.00
defect                 1   2           3        2       1      66.67     33.33
dependency             2   0           2        0       2       0.00    100.00
how-to                 0   2           2        2       0     100.00      0.00
workaround             1   1           2        1       1      50.00     50.00
low-internal-quality   0   1           1        1       0     100.00      0.00
requirement            1   0           1        0       1       0.00    100.00
superficial-test       1   0           1        0       1       0.00    100.00

 ##########detect_trained-bert-5fcv-1##########
confusion             FN  TP  total_satd  correct  failed  correct_%  failed_%
type                     

RQ3 False Negative (FN) Analysis

In [2]:
import pandas as pd
import os

INPUT_FILE = "../result/rq3/unique/detect_flan-t5-xl-mat-0-shot.csv"
OUTPUT_FILE = "../cache/output/analysis/detect_flan-t5-xl-mat-0-shot_random_fn.csv"

os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)


# Load detection results
df = pd.read_csv(INPUT_FILE)

# Select False Negatives (SATD predicted as non-SATD)
fn_df = df[(df["label"] != "no") & (df["label_pred"] == "no")]
print(f"False Negatives: {len(fn_df)}/{len(df)}")

# For 95% confidence of 61 samples is 53 so take the whole
IDEAL_SAMPLE_SIZE = len(fn_df)



# Random sample
sample_df = fn_df.sample(
    n=min(IDEAL_SAMPLE_SIZE, len(fn_df)),
    random_state=42
)

# Save to CSV
sample_df.to_csv(OUTPUT_FILE, index=False)

False Negatives: 61/6531


RQ4 False Positive (FP) Analysis

In [3]:
import pandas as pd
import os

INPUT_FILE = "../result/rq4/unique/detect_gpt-5-2-shot.csv"
OUTPUT_FILE = "../cache/output/analysis/detect_gpt-5-2-shot_random_fp.csv"
os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

# Load detection results
df = pd.read_csv(INPUT_FILE)

# Select False Positives (non-SATD predicted as SATD)
fp_df = df[(df["label"] == "no") & (df["label_pred"] == "yes")]
print(f"False Positives: {len(fp_df)}/{len(df)}")

# For 95% confidence of 233 samples is 146
IDEAL_SAMPLE_SIZE = 146



# Random sample
sample_df = fp_df.sample(n=min(IDEAL_SAMPLE_SIZE, len(fp_df)), random_state=42)

# Save to CSV
sample_df.to_csv(OUTPUT_FILE, index=False)


False Positives: 233/6531
